In [2]:
import cutlass
import cutlass.cute as cute


In [5]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, message="CUDA_TOOLKIT_PATH environment variable is not set.*")

In [2]:
@cute.kernel
def hello_kernel():
    tidx, _, _ = cute.arch.thread_idx()
    if tidx == 0:
        cute.printf("Hello from GPU")

@cute.jit
def hello_world():
    cutlass.cuda.initialize_cuda_context()
    hello_kernel().launch(grid=(1, 1, 1), block=(32, 1, 1))

In [ ]:
compiled = cute.compile(hello_world)
compiled()

/home/warmonkeys/miniconda3/envs/cutedsl/lib/python3.12/site-packages/nvidia_cutlass_dsl/python_packages/cutlass/base_dsl/dsl.py:412: UserWarning: CUDA_TOOLKIT_PATH environment variable is not set. Cannot set toolkitPath.
  warnings.warn(message, UserWarning)


0

Hello from GPU


In [4]:
@cute.jit
def print_demo(a: cutlass.Int32, b: cutlass.Constexpr[int]):
    print("static a:", a)   # => ? (dynamic)
    print("static b:", b)   # => 2
    cute.printf("dynamic a: {}", a)
    cute.printf("dynamic b: {}", b)
    layout = cute.make_layout((a, b))
    print("static layout:", layout)       # (?,2):(1,?)
    cute.printf("dynamic layout: {}", layout)  # (8,2):(1,8)

In [5]:
print_demo(cutlass.Int32(8), 2)

static a: ?
static b: 2
static layout: (?,2):(1,?)
dynamic a: 8
dynamic b: 2
dynamic layout: (8,2):(1,8)


In [6]:
@cute.jit
def dtypes():
    a = cutlass.Int32(42)
    b = a.to(cutlass.Float32)
    c = b + 0.5
    d = c.to(cutlass.Int32)
    cute.printf("a={}, b={}, c={}, d={}", a, b, c, d)

dtypes()

a=42, b=42.000000, c=42.500000, d=42


In [3]:
import torch
from cutlass.cute.runtime import from_dlpack

In [4]:
@cute.jit
def tensor_demo(t: cute.Tensor):
    cute.printf("t[0,0] = {}", t[0, 0])
    sub = t[(None, 0)]   # First row view
    frag = cute.make_fragment(sub.layout, sub.element_type)
    frag.store(sub.load())
    cute.print_tensor(frag)

arr = torch.arange(0, 12, dtype=torch.float32).reshape(3, 4)
tensor_demo(from_dlpack(arr))

t[0,0] = 0.000000
tensor(raw_ptr(0x00007ffe5973fa80: f32, rmem, align<32>) o (3):(4), data=
       [ 0.000000, ],
       [ 4.000000, ],
       [ 8.000000, ])


/home/warmonkeys/miniconda3/envs/cutedsl/lib/python3.12/site-packages/nvidia_cutlass_dsl/python_packages/cutlass/base_dsl/dsl.py:412: UserWarning: CUDA_TOOLKIT_PATH environment variable is not set. Cannot set toolkitPath.
  warnings.warn(message, UserWarning)


In [5]:
@cute.jit
def layout_stride_demo(M: cutlass.Int32, N: cutlass.Int32):
    row_major = cute.make_layout((M, N), stride=(N, cutlass.Int32(1)))
    col_major = cute.make_layout((M, N), stride=(cutlass.Int32(1), M))
    print("static row-major:", row_major)
    print("static col-major:", col_major)
    cute.printf("dynamic row-major: {}", row_major)
    cute.printf("dynamic col-major: {}", col_major)

layout_stride_demo(cutlass.Int32(4), cutlass.Int32(3))

static row-major: (?,?):(?,?)
static col-major: (?,?):(?,?)
dynamic row-major: (4,3):(3,1)
dynamic col-major: (4,3):(1,4)


In [9]:
@cute.jit
def slicing_examples(t: cute.Tensor):
    # Scalar access
    cute.printf("t[1,2] = {}", t[1, 2])

    # Entire second row (shape: (N,)) using (None, row_index)
    row = t[(None, 1)]
    row_frag = cute.make_fragment(row.layout, row.element_type)
    row_frag.store(row.load())
    print("Second row:")
    cute.print_tensor(row_frag)

    # Entire third column (shape: (M,)) using (col_index, None)
    col = t[(2, None)]
    col_frag = cute.make_fragment(col.layout, col.element_type)
    col_frag.store(col.load())
    print("Third column:")
    cute.print_tensor(col_frag)

    # Printing the first row directly (*t[2] == *t[2, 0])
    cute.printf(
        "t[2] = {} (equivalent to t[{}])",
        t[2],
        cute.make_identity_tensor(t.layout.shape)[2]
    )
    cute.printf(cute.make_identity_tensor(t.layout.shape))

# 4x3 example tensor
arr = torch.arange(12, dtype=torch.float32).reshape(4, 3)
slicing_examples(from_dlpack(arr))

Second row:
Third column:
t[1,2] = 5.000000
tensor(raw_ptr(0x00007ffe5973fa80: f32, rmem, align<32>) o (4):(3), data=
       [ 1.000000, ],
       [ 4.000000, ],
       [ 7.000000, ],
       [ 10.000000, ])
tensor(raw_ptr(0x00007ffe5973fa60: f32, rmem, align<32>) o (3):(1), data=
       [ 6.000000, ],
       [ 7.000000, ],
       [ 8.000000, ])
t[2] = 6.000000 (equivalent to t[(2,0)])
(0,0) o (4,3):(1@0,1@1)


In [4]:
import numpy as np

In [4]:
@cute.jit
def ssa_add(dst: cute.Tensor, x: cute.Tensor, y: cute.Tensor):
    xv = x.load()
    yv = y.load()
    dst.store(xv + yv)
    cute.print_tensor(dst)

X = np.ones((2, 3), dtype=np.float32)
Y = np.full((2, 3), 2.0, dtype=np.float32)
Z = np.zeros((2, 3), dtype=np.float32)
ssa_add(from_dlpack(Z), from_dlpack(X), from_dlpack(Y))

tensor(raw_ptr(0x000000003f3603c0: f32, generic, align<4>) o (2,3):(3,1), data=
       [[ 3.000000,  3.000000,  3.000000, ],
        [ 3.000000,  3.000000,  3.000000, ]])


/home/warmonkeys/miniconda3/envs/cutedsl/lib/python3.12/site-packages/nvidia_cutlass_dsl/python_packages/cutlass/base_dsl/dsl.py:412: UserWarning: CUDA_TOOLKIT_PATH environment variable is not set. Cannot set toolkitPath.
  warnings.warn(message, UserWarning)


In [5]:
@cute.jit
def ssa_reduce(a: cute.Tensor):
    v = a.load()
    # Sum of all elements
    total = v.reduce(cute.ReductionOp.ADD, 0.0, reduction_profile=0)
    cute.printf("total sum = {}", total)

    # Row-wise sum -> shape (rows,)
    row_sum = v.reduce(cute.ReductionOp.ADD, 0.0, reduction_profile=(None, 1))
    row_frag = cute.make_fragment(row_sum.shape, cutlass.Float32)
    row_frag.store(row_sum)
    print("Row-wise sum:")
    cute.print_tensor(row_frag)

    # Column-wise sum -> shape (cols,)
    col_sum = v.reduce(cute.ReductionOp.ADD, 0.0, reduction_profile=(1, None))
    col_frag = cute.make_fragment(col_sum.shape, cutlass.Float32)
    col_frag.store(col_sum)
    print("Column-wise sum:")
    cute.print_tensor(col_frag)

A = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float32)
ssa_reduce(from_dlpack(A))

Row-wise sum:
Column-wise sum:
total sum = 21.000000
tensor(raw_ptr(0x00007ffe0065f620: f32, rmem, align<32>) o (2):(1), data=
       [ 6.000000, ],
       [ 15.000000, ])
tensor(raw_ptr(0x00007ffe0065f640: f32, rmem, align<32>) o (3):(1), data=
       [ 5.000000, ],
       [ 7.000000, ],
       [ 9.000000, ])


In [6]:
@cute.kernel
def vadd_kernel(gA: cute.Tensor, gB: cute.Tensor, gC: cute.Tensor):
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()
    bdim, _, _ = cute.arch.block_dim()
    idx = bidx * bdim + tidx
    m, n = gA.shape[1]          # thread-domain
    mi = idx // n
    ni = idx % n
    gC[(None, (mi, ni))] = gA[(None, (mi, ni))].load() + gB[(None, (mi, ni))].load()

@cute.jit
def vadd(A: cute.Tensor, B: cute.Tensor, C: cute.Tensor):
    gA = cute.zipped_divide(A, (1, 4))
    gB = cute.zipped_divide(B, (1, 4))
    gC = cute.zipped_divide(C, (1, 4))
    threads = 256
    vadd_kernel(gA, gB, gC).launch(
        grid=(cute.size(gC, mode=[1]) // threads, 1, 1),
        block=(threads, 1, 1),
    )

M, N = 1024, 1024
a = torch.randn(M, N, device="cuda", dtype=torch.float16)
b = torch.randn(M, N, device="cuda", dtype=torch.float16)
c = torch.zeros(M, N, device="cuda", dtype=torch.float16)
vadd_compiled = cute.compile(vadd, from_dlpack(a), from_dlpack(b), from_dlpack(c))
vadd_compiled(from_dlpack(a), from_dlpack(b), from_dlpack(c))

0

In [7]:
@cute.jit
def zdiv_demo(mA: cute.Tensor):
    # Partition into per-thread tiles of (1,4)
    gA = cute.zipped_divide(mA, (1, 4))
    print("Tiled tensor gA:", gA)

    # Inspect a specific tile (mi, ni)
    mi = cutlass.Int32(0)
    ni = cutlass.Int32(0)
    tile = gA[(None, (mi, ni))]
    print("Per-thread tile slice:", tile)

    # Materialize tile for printing
    frag = cute.make_fragment(tile.layout, tile.element_type)
    frag.store(tile.load())
    cute.print_tensor(frag)

A = torch.arange(0, 8*8, dtype=torch.float32).reshape(8, 8)
zdiv_demo(from_dlpack(A))

Tiled tensor gA: tensor<ptr<f32, generic> o ((1,4),(8,2)):((0,1),(8,4))>
Per-thread tile slice: tensor<ptr<f32, generic> o ((1,4)):((0,1))>
tensor(raw_ptr(0x00007ffe0065f6c0: f32, rmem, align<32>) o ((1,4)):((0,1)), data=
       [ 0.000000, ],
       [ 1.000000, ],
       [ 2.000000, ],
       [ 3.000000, ])


In [ ]:
@cute.kernel
def tv_add_kernel(gA: cute.Tensor, gB: cute.Tensor, gC: cute.Tensor, tv_layout: cute.Layout):
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()

    # Select the thread-block tile
    #select all of values (none, none), bidx is selecting which block within grid, bidx ranges is 1d
    blk_coord = ((None, None), bidx)
    blkA = gA[blk_coord]
    blkB = gB[blk_coord]
    blkC = gC[blk_coord]
    #blkA is size 16,256

    # Compose TV layout to map (tid, vid) -> physical address
    tidfrgA = cute.composition(blkA, tv_layout)
    tidfrgB = cute.composition(blkB, tv_layout)
    tidfrgC = cute.composition(blkC, tv_layout)
    #reindex blkA, to tid, vid. Because we need to only process the tid portion

    # Slice per-thread vector
    thr_coord = (tidx, None)
    thrA = tidfrgA[thr_coord]
    thrB = tidfrgB[thr_coord]
    thrC = tidfrgC[thr_coord]

    thrC[None] = thrA.load() + thrB.load()

@cute.jit
def tv_add(mA: cute.Tensor, mB: cute.Tensor, mC: cute.Tensor):
    # Thread (4,32): 4 groups along M (row), 32 contiguous threads along N (col)
    # Value (4,8): each thread handles 4 rows x 8 contiguous values
    thr_layout = cute.make_layout((4, 32), stride=(32, 1))
    val_layout = cute.make_layout((4, 8), stride=(8, 1))
    #tiler_mn is tensor of shape 16, 256. tv_layout will be used to convert tid, vid -> (tile m, tile n)
    #tid ranges 0-128 (4x32), vid runs 32 values (4x8), tile_m ranges from 0-15, and tile_n ranges 0-255
    tiler_mn, tv_layout = cute.make_layout_tv(thr_layout, val_layout)

    # Tile tensors into thread-block tiles
    #gA mode 0 - size 16, 256, mode 1 - size 128 x 8
    gA = cute.zipped_divide(mA, tiler_mn)
    gB = cute.zipped_divide(mB, tiler_mn)
    gC = cute.zipped_divide(mC, tiler_mn)

    # One block per tile in mode-1; threads per block = TV threads
    tv_add_kernel(gA, gB, gC, tv_layout).launch(
        grid=[cute.size(gC, mode=[1]), 1, 1],
        block=[cute.size(tv_layout, mode=[0]), 1, 1],
    )

M, N = 2048, 2048
a = torch.randn(M, N, device="cuda", dtype=torch.float16)
b = torch.randn(M, N, device="cuda", dtype=torch.float16)
c = torch.zeros(M, N, device="cuda", dtype=torch.float16)
tv_add_compiled = cute.compile(tv_add, from_dlpack(a), from_dlpack(b), from_dlpack(c))
tv_add_compiled(from_dlpack(a), from_dlpack(b), from_dlpack(c))

/home/warmonkeys/miniconda3/envs/cutedsl/lib/python3.12/site-packages/nvidia_cutlass_dsl/python_packages/cutlass/base_dsl/dsl.py:412: UserWarning: CUDA_TOOLKIT_PATH environment variable is not set. Cannot set toolkitPath.
  warnings.warn(message, UserWarning)


0

In [ ]:
@cute.jit
def layout_demo():
    # Composition and coalesce
    #basically, Composition (A,B) means B represents the domain of R, and A maps takes B as input and maps to some address. Usually, A distinct entry from (4,3) maps to a specific entry in ((2,2,),3). Resulting shape is determined by cutedsl.
    A = cute.make_layout((6, 2), stride=(cutlass.Int32(8), 2))
    B = cute.make_layout((4, 3), stride=(3, 1))
    R = cute.composition(A, B)
    C = cute.coalesce(R)

    # Logical divide with tiler
    #Difficult to understand, the first dimension/tile division is self explanatory. Second one is 2d tile division. (4,8) divided by tile (2,4) but the stride of the tile is (1,8), as opposed to row major stride (4,1). Suppose it was row major stride, then the stride within the tile would just be (1,13)? But because stride is (1,8), then the stride is (13,2). reason being 1, going down the row, you index +1 in the tile, but the stride going to the side is +8, but since adding 8 to the index, with only 4 items per row, that means you skip 2 rows, and end up 2 rows down (thus +2). Then, in the actual parent tile itself, the stride downwards is 13, and the stride sideways is 1. Then, for the switch to the next tile, the stride is (26,1). Downwards you move 2 indices (each of stride 13). But sideways you move 1, because the first tile would have covered all even columns, and the second tile would cover all odd columns.
    L = cute.make_layout((9, (4, 8)), stride=(59, (13, 1)))
    T = (cute.make_layout(3, stride=3),
         cute.make_layout((2, 4), stride=(1, 8)))
    D = cute.logical_divide(L, tiler=T)

    # Logical product/repetition
    P = cute.logical_product(
        cute.make_layout((2, 2), stride=(4, 1)),
        cute.make_layout(6, stride=1),
    )

    cute.printf("A={}, B={}, R={}, C={}", A, B, R, C)
    cute.printf("Divide: {}", D)
    cute.printf("Product: {}", P)

layout_demo()

A=(6,2):(8,2), B=(4,3):(3,1), R=((2,2),3):((24,2),8), C=(2,2,3):(24,2,8)
Divide: ((3,3),((2,4),(2,2))):((177,59),((13,2),(26,1)))
Product: ((2,2),(2,3)):((4,1),(2,8))


/home/warmonkeys/miniconda3/envs/cutedsl/lib/python3.12/site-packages/nvidia_cutlass_dsl/python_packages/cutlass/base_dsl/dsl.py:412: UserWarning: CUDA_TOOLKIT_PATH environment variable is not set. Cannot set toolkitPath.
  warnings.warn(message, UserWarning)
